In [2]:
import os
import pandas as pd
import numpy as np
import duckdb

In [ ]:
path_dataframe = 'data/2023_Yellow_Taxi_Trip_Data_20260526.csv'

# CONVERSÃO PARA PARQUET E CONVERSÃO DE TIPOS

In [ ]:
duckdb.sql("""
COPY (
    SELECT *
    FROM read_csv_auto(
        path_dataframe,
        all_varchar=true
    )
)
TO 'data/taxi_raw.parquet'
(FORMAT PARQUET)
""")

In [13]:
df = duckdb.sql("""
SELECT *
FROM read_parquet('data/taxi_raw.parquet')
LIMIT 5
""").df()

print(df.columns.tolist())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [14]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_clean AS
SELECT
    VendorID::INTEGER AS VendorID,

    STRPTIME(
        tpep_pickup_datetime,
        '%m/%d/%Y %I:%M:%S %p'
    ) AS tpep_pickup_datetime,

    STRPTIME(
        tpep_dropoff_datetime,
        '%m/%d/%Y %I:%M:%S %p'
    ) AS tpep_dropoff_datetime,

    passenger_count::INTEGER AS passenger_count,

    REPLACE(trip_distance, ',', '')::DOUBLE AS trip_distance,

    RatecodeID::INTEGER AS RatecodeID,

    store_and_fwd_flag,

    PULocationID::INTEGER AS PULocationID,
    DOLocationID::INTEGER AS DOLocationID,

    payment_type::INTEGER AS payment_type,

    REPLACE(fare_amount, ',', '')::DOUBLE AS fare_amount,
    REPLACE(extra, ',', '')::DOUBLE AS extra,
    REPLACE(mta_tax, ',', '')::DOUBLE AS mta_tax,
    REPLACE(tip_amount, ',', '')::DOUBLE AS tip_amount,
    REPLACE(tolls_amount, ',', '')::DOUBLE AS tolls_amount,
    REPLACE(improvement_surcharge, ',', '')::DOUBLE AS improvement_surcharge,
    REPLACE(total_amount, ',', '')::DOUBLE AS total_amount,
    REPLACE(congestion_surcharge, ',', '')::DOUBLE AS congestion_surcharge,
    REPLACE(airport_fee, ',', '')::DOUBLE AS airport_fee

FROM read_parquet('data/taxi_raw.parquet')
""")

In [15]:
duckdb.sql("""
COPY taxi_clean
TO 'data/taxi.parquet'
(FORMAT PARQUET)
""")

# TRATAMENTO DOS DADOS

Usecols = hour, tpep_pickup_time, PULocationID, DOLocationID
Dropna

In [26]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand AS

SELECT
    DATE_TRUNC(
        'hour',
        tpep_pickup_datetime
    ) AS pickup_hour,
    
    PULocationID,
    COUNT(*) AS demand

FROM 'data/taxi.parquet'

WHERE
    tpep_pickup_datetime IS NOT NULL
    AND PULocationID IS NOT NULL
    
    AND YEAR(tpep_pickup_datetime) = 2023

GROUP BY
    pickup_hour,
    PULocationID

ORDER BY
    pickup_hour,
    PULocationID
""")

Criação de colunas descritivas a partir das colunas selecionadas previamente
novas_colunas = year, month, day, hour, weekday, is_weekend, covid_period

In [27]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand_features AS

SELECT
    pickup_hour,
    PULocationID,
    demand,
    MONTH(pickup_hour) AS month,
    DAY(pickup_hour) AS day,
    HOUR(pickup_hour) AS hour,
    DAYOFWEEK(pickup_hour) AS weekday,
    CASE
        WHEN DAYOFWEEK(pickup_hour) IN (0, 6)
            THEN 1
        ELSE 0
    END AS is_weekend,

FROM taxi_demand

ORDER BY
    pickup_hour,
    PULocationID
""")

Salva o parquet final de demanda

In [ ]:
duckdb.sql("""
COPY taxi_demand_features
TO 'data/taxi_demand_processed_pu.parquet'
(FORMAT PARQUET)
""")

In [ ]:
print(
    duckdb.sql("""
    SELECT COUNT(*) AS total_rows
    FROM 'data/taxi_demand_processed_pu.parquet'
    """).fetchall()
)

print(
    duckdb.sql("""
    SELECT *
    FROM 'data/taxi_demand_processed_pu.parquet'
    LIMIT 10
    """).df()
)

[(886277,)]
  pickup_hour  PULocationID  demand  month  day  hour  weekday  is_weekend
0  2023-01-01             4      19      1    1     0        0           1
1  2023-01-01             7       3      1    1     0        0           1
2  2023-01-01            12       1      1    1     0        0           1
3  2023-01-01            13      14      1    1     0        0           1
4  2023-01-01            24      20      1    1     0        0           1
5  2023-01-01            25       4      1    1     0        0           1
6  2023-01-01            33      12      1    1     0        0           1
7  2023-01-01            36       6      1    1     0        0           1
8  2023-01-01            37       1      1    1     0        0           1
9  2023-01-01            39       1      1    1     0        0           1


In [ ]:
print(
    duckdb.sql("""
    SELECT
        COUNT(*) AS n,
        MIN(demand) AS min_demand,
        MAX(demand) AS max_demand,
        AVG(demand) AS mean_demand,
        STDDEV(demand) AS std_demand,
        MEDIAN(demand) AS median_demand
    FROM 'data/taxi_demand_processed_pu.parquet'
    """).df()
)

        n  min_demand  max_demand  mean_demand  std_demand  median_demand
0  886277           1        1106    43.225901    75.43203            6.0
